In [3]:
import pandas as pd
import numpy as np

# Loading Dataset

In [4]:
interactions = pd.read_csv("processed_interactions.csv")
interactions[["user_id", "business_id", "stars", "date"]].head()

,user_id,business_id,stars,date
0,-FxsSuwDbIII7yo5BjHpiA,sr-5EY6bmp4jINhea06MjA,4,2016-06-15 18:24:36
1,-FxsSuwDbIII7yo5BjHpiA,Gf1EboxqdJ9SPsVJur93Cw,4,2016-08-26 14:06:32
2,-FxsSuwDbIII7yo5BjHpiA,yeHLiKNp0hyR-ig4M6us-w,5,2016-11-22 15:26:03
3,-FxsSuwDbIII7yo5BjHpiA,5FDt7sy-70y4_37Dh2qcbw,4,2017-03-15 19:41:39
4,-FxsSuwDbIII7yo5BjHpiA,3y61A28PQDZ4weeeTRfIlA,3,2017-10-05 20:55:17


Rating thresholding (matches Section 4.2.5)

In [5]:
interactions["label"] = (interactions["stars"] >= 3).astype(int)

Temporal Sorting

In [6]:
interactions["date"] = pd.to_datetime(interactions["date"])
interactions = interactions.sort_values(["user_id", "date"])

Ratio-based per-user split (80/10/10), matching Section 4.4 to 4.4.2

In [7]:
def split_user(df):
    n = len(df)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    
    return (
        df.iloc[:train_end],
        df.iloc[train_end:val_end],
        df.iloc[val_end:]
    )

train, val, test = [], [], []

for _, user_df in interactions.groupby("user_id"):
    if len(user_df) < 5:
        continue
    tr, va, te = split_user(user_df)
    train.append(tr)
    val.append(va)
    test.append(te)

train_df = pd.concat(train)
val_df = pd.concat(val)
test_df = pd.concat(test)

In [8]:
print("Users:", train_df["user_id"].nunique())
print("Items:", train_df["business_id"].nunique())
print("Interactions:", len(train_df))
print("Avg interactions/user:", len(train_df) / train_df["user_id"].nunique())

Users: 413
Items: 866
Interactions: 2044
Avg interactions/user: 4.9491525423728815


In [14]:
import sys
sys.version

'3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]'

In [2]:
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.evaluation.python_evaluation import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k
)

print("NeuCF imports OK")


NeuCF imports OK


In [9]:
def to_recommenders_format(df):
    return df.rename(columns={
        "user_id": "userID",
        "business_id": "itemID",
        "label": "label",
        "date": "timestamp"
    })[["userID", "itemID", "label", "timestamp"]]

train_rec = to_recommenders_format(train_df)
val_rec   = to_recommenders_format(val_df)
test_rec  = to_recommenders_format(test_df)

# Matching hyperparameters exactly

## GMF

In [20]:
# Create mappings from training data
user2id = {u: i for i, u in enumerate(train_df["user_id"].unique())}
item2id = {i: j for j, i in enumerate(train_df["business_id"].unique())}

In [21]:
def apply_mapping(df, user2id, item2id):
    df = df.copy()
    df["userID"] = df["user_id"].map(user2id)
    df["itemID"] = df["business_id"].map(item2id)
    return df.dropna(subset=["userID", "itemID"])

In [22]:
train_rec = apply_mapping(train_df, user2id, item2id)
val_rec   = apply_mapping(val_df, user2id, item2id)
test_rec  = apply_mapping(test_df, user2id, item2id)

train_rec["userID"] = train_rec["userID"].astype(int)
train_rec["itemID"] = train_rec["itemID"].astype(int)
val_rec["userID"]   = val_rec["userID"].astype(int)
val_rec["itemID"]   = val_rec["itemID"].astype(int)
test_rec["userID"]  = test_rec["userID"].astype(int)
test_rec["itemID"]  = test_rec["itemID"].astype(int)

In [23]:
train_rec[["userID", "itemID", "label"]].to_csv("train.csv", index=False)
val_rec[["userID", "itemID", "label"]].to_csv("val.csv", index=False)
test_rec[["userID", "itemID", "label"]].to_csv("test.csv", index=False)

In [33]:
from recommenders.models.ncf.dataset import Dataset

dataset = Dataset(
    train_file="train.csv",
    test_file="test.csv",      # optional but recommended
    col_user="userID",
    col_item="itemID",
    col_rating="label",
    binary=True,
    n_neg=4,                   # default in NCF paper
    n_neg_test=100,             # standard for evaluation
    seed=42
)

INFO:recommenders.models.ncf.dataset:Indexing train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing test.csv ...
INFO:recommenders.models.ncf.dataset:Creating full leave-one-out test file test_full.csv ...
  0%|          | 0/280 [00:00<?, ?it/s]c:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\venv\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\venv\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
c:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\venv\lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is dep

In [34]:
dataset.n_users
dataset.n_items

866

In [46]:
from recommenders.models.ncf.ncf_singlenode import NCF

gmf_dir = "checkpoints/gmf"

gmf = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="GMF",
    n_factors=1024,
    learning_rate=1e-3,
    batch_size=8192,
    n_epochs=100,
    verbose=1,
    seed=42
)

gmf.fit(dataset)
gmf.save(gmf_dir)

c:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\venv\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1697: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 1 [0.47s]: train_loss = 0.693146 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 2 [0.56s]: train_loss = 0.693143 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 3 [0.40s]: train_loss = 0.693138 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 4 [0.41s]: train_loss = 0.693134 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 5 [0.48s]: train_loss = 0.693128 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 6 [1.14s]: train_loss = 0.693121 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 7 [1.25s]: train_loss = 0.693114 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 8 [1.17s]: train_loss = 0.693107 


In [47]:
from recommenders.models.ncf.ncf_singlenode import NCF

mlp_dir = "checkpoints/mlp"

mlp = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="MLP",
    n_factors=1024,
    layer_sizes=[8],
    learning_rate=1e-3,
    batch_size=8192,
    n_epochs=100,
    verbose=1,
    seed=42
)

mlp.fit(dataset)
mlp.save(mlp_dir)

INFO:recommenders.models.ncf.ncf_singlenode:Epoch 1 [0.40s]: train_loss = 0.693083 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 2 [0.35s]: train_loss = 0.691911 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 3 [0.73s]: train_loss = 0.690900 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 4 [0.39s]: train_loss = 0.689848 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 5 [0.34s]: train_loss = 0.688829 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 6 [0.73s]: train_loss = 0.687863 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 7 [0.98s]: train_loss = 0.686707 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 8 [1.38s]: train_loss = 0.685730 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 9 [1.36s]: train_loss = 0.684654 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [1.12s]: train_loss = 0.683546 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 11 [0.50s]: train_loss = 0.682489 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 12 [0.36s]: train_loss =

In [48]:
neucf = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="NeuMF",
    n_factors=1024,
    layer_sizes=[8],
    learning_rate=1e-2,
    batch_size=8192,
    n_epochs=100,
    verbose=1,
    seed=42
)

In [49]:
neucf._load_neumf(
    gmf_dir=gmf_dir,
    mlp_dir=mlp_dir,
    alpha=0.5
)

INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/mlp\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/mlp\model.ckpt


In [50]:
neucf.fit(dataset)

INFO:recommenders.models.ncf.ncf_singlenode:Epoch 1 [0.62s]: train_loss = 0.485600 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 2 [0.42s]: train_loss = 0.444198 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 3 [0.42s]: train_loss = 0.418623 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 4 [0.40s]: train_loss = 0.402259 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 5 [0.58s]: train_loss = 0.399731 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 6 [0.45s]: train_loss = 0.397691 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 7 [1.17s]: train_loss = 0.389651 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 8 [1.27s]: train_loss = 0.395104 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 9 [1.53s]: train_loss = 0.385964 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [0.64s]: train_loss = 0.375484 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 11 [0.48s]: train_loss = 0.366663 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 12 [0.60s]: train_loss =

In [51]:
neucf.save("checkpoints/neucf")

# Generate predictions

In [1]:
import numpy as np
import pandas as pd

from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset

In [6]:
dataset = Dataset(
    train_file="train.csv",
    test_file="test.csv",
    n_neg=4,
    n_neg_test=100,
    col_user="userID",
    col_item="itemID",
    col_rating="label",
    binary=True,
    seed=42
)

INFO:recommenders.models.ncf.dataset:Indexing train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing test.csv ...
INFO:recommenders.models.ncf.dataset:Indexing test_full.csv ...


In [3]:
gmf = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="GMF",
    n_factors=1024,
    learning_rate=1e-3,
    batch_size=8192,
    n_epochs=100,
    verbose=0
)
gmf.load("checkpoints/gmf")

c:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\venv\lib\site-packages\tensorflow\python\keras\engine\base_layer_v1.py:1697: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '



INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


In [7]:
mlp = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="MLP",
    n_factors=1024,
    layer_sizes=[8],
    learning_rate=1e-3,
    batch_size=8192,
    n_epochs=100,
    verbose=0,
    seed=42
)

mlp.fit(dataset)

In [8]:
neucf = NCF(
    n_users=dataset.n_users,
    n_items=dataset.n_items,
    model_type="NeuMF",
    n_factors=1024,
    layer_sizes=[8],
    learning_rate=1e-2,   # Appendix A.1.3
    batch_size=8192,
    n_epochs=100,
    verbose=0,
    seed=42
)

neucf.load(
    gmf_dir="checkpoints/gmf",
    mlp_dir="checkpoints/mlp",
    alpha=0.5
)

INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/gmf\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/mlp\model.ckpt


INFO:tensorflow:Restoring parameters from checkpoints/mlp\model.ckpt


In [11]:
def attach_dataset_mappings(model, dataset):
    model.user2id = dataset.user2id
    model.item2id = dataset.item2id
    model.id2user = dataset.id2user
    model.id2item = dataset.id2item

attach_dataset_mappings(gmf, dataset)
attach_dataset_mappings(mlp, dataset)
attach_dataset_mappings(neucf, dataset)

In [17]:
import numpy as np

def evaluate_ncf(model, dataset, k=10):
    hits, ndcgs = [], []

    for users, items, labels in dataset.test_loader(yield_id=False):
        scores = model.predict(users, items, is_list=True)

        scores = np.array(scores)
        labels = np.array(labels)

        # rank items by score
        order = np.argsort(scores)[::-1]
        ranked_labels = labels[order][:k]

        # HR@k
        hit = int(ranked_labels.sum() > 0)
        hits.append(hit)

        # NDCG@k
        if hit:
            pos = np.where(ranked_labels == 1)[0][0]
            ndcgs.append(1 / np.log2(pos + 2))
        else:
            ndcgs.append(0)

    hr = np.mean(hits)
    ndcg = np.mean(ndcgs)

    # Recall@10 == HR@10 in leave-one-out
    return hr, hr, ndcg

In [18]:
results = []

for name, model in [
    ("GMF", gmf),
    ("MLP", mlp),
    ("NeuCF", neucf),
]:
    p, r, n = evaluate_ncf(model, dataset, k=10)
    results.append({
        "Model": name,
        "Precision@10": round(p, 4),
        "Recall@10": round(r, 4),
        "NDCG@10": round(n, 4),
    })

table_5_1 = pd.DataFrame(results)
table_5_1

,Model,Precision@10,Recall@10,NDCG@10
0,GMF,0.2500,0.2500,0.1329
1,MLP,0.2639,0.2639,0.1322
2,NeuCF,0.2535,0.2535,0.1337


# Sanity Check

In [23]:
from recommenders.models.ncf.dataset import Dataset

dataset = Dataset(
    train_file="train.csv",
    test_file="test.csv",          # leave-one-out positives
    test_file_full="test_full.csv",# auto-generated negatives
    n_neg_test=100,
    col_user="userID",
    col_item="itemID",
    col_rating="label",
    binary=True,
    seed=42
)

INFO:recommenders.models.ncf.dataset:Indexing train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing test.csv ...
INFO:recommenders.models.ncf.dataset:Indexing test_full.csv ...


In [24]:
users, items, labels = next(dataset.test_loader(yield_id=False))

print(len(users))          # MUST be 101
print(sum(labels))         # MUST be 1

101
1.0


In [25]:
import numpy as np

def evaluate_ncf(model, dataset, k=10):
    precisions, recalls, ndcgs = [], [], []

    for users, items, labels in dataset.test_loader(yield_id=False):
        scores = []
        for u, i in zip(users, items):
            scores.append(model.predict(u, i))

        scores = np.array(scores)
        labels = np.array(labels)

        topk = np.argsort(scores)[::-1][:k]
        hits = labels[topk]

        precisions.append(hits.sum() / k)
        recalls.append(1.0 if hits.sum() > 0 else 0.0)

        if hits.sum() > 0:
            rank = np.where(hits == 1)[0][0] + 1
            ndcgs.append(1 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

In [26]:
results = []

for name, model in [
    ("GMF", gmf),
    ("MLP", mlp),
    ("NeuCF", neucf),
]:
    p, r, n = evaluate_ncf(model, dataset, k=10)
    results.append({
        "Model": name,
        "Precision@10": round(p, 4),
        "Recall@10": round(r, 4),
        "NDCG@10": round(n, 4),
    })

table_5_1 = pd.DataFrame(results)
table_5_1

,Model,Precision@10,Recall@10,NDCG@10
0,GMF,0.0250,0.2500,0.1329
1,MLP,0.0264,0.2639,0.1322
2,NeuCF,0.0253,0.2535,0.1337
